# Extra Session: Meridian with Geo-Level Weekly Data

In Session 5 we fitted Meridian on **36 monthly, national-level** observations — the bare minimum for an MMM.
Here we use a richer dataset: **5 US regions × 104 weeks = 520 data points**. This unlocks three
capabilities that national monthly data cannot provide:

| Dimension | Session 5 (national monthly) | This notebook (geo weekly) |
|-----------|----------------------------|---------------------------|
| Time granularity | Monthly (36 periods) | Weekly (104 periods) |
| Geographic scope | 1 national aggregate | 5 US regions |
| Total observations | 36 | 520 |
| Adstock resolution | Coarse (monthly carry-over) | Fine (weekly decay curves) |
| Hierarchical modeling | Not applicable | Geo-level effects share information |
| Causal identification | Time-series variation only | Geo × time variation |

### Why does this matter?

1. **Better adstock estimation** — Monthly data blurs short-lived effects (e.g., a paid-search click
   that converts within days). Weekly data preserves these dynamics so Meridian can distinguish
   fast-decaying channels (search) from slow-decaying ones (TV).

2. **Hierarchical pooling** — Meridian treats each geo's media coefficient as a draw from a shared
   national distribution. Regions with sparse data borrow strength from data-rich regions, producing
   more stable estimates than fitting each geo independently.

3. **Stronger causal identification** — When TV spend varies across geos and weeks, the model can
   separate TV's effect from seasonality more cleanly than when it only sees one national time series.
   Geographic variation acts as quasi-experimental data.

By the end of this notebook you will have a fitted geo-level Meridian model and be able to compare
ROI, response curves, and contributions at both the national and regional level.

In [ ]:
# --- Setup ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Meridian imports
try:
    import meridian
    from meridian.data.input_data import InputData
    from meridian.model import model as meridian_model
    from meridian.model import spec as meridian_spec
    from meridian.analysis.analyzer import Analyzer
    import xarray as xr
    MERIDIAN_AVAILABLE = True
    print(f"Meridian {meridian.__version__} loaded successfully.")
except ImportError:
    MERIDIAN_AVAILABLE = False
    print("Meridian not installed. See Notebook 4 for setup instructions.")
    print("You can still follow along — all outputs are saved in the notebook.")

# Load geo-level weekly data
df = pd.read_csv('../../data/synthetic_multi_geo_data.csv')
print(f"\nDataset: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Geos: {sorted(df['geo'].unique())}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
df.head()

## Data Exploration

Before building the model, let's understand the dataset structure. The synthetic data
simulates a brand operating across five US regions with five paid media channels, each
having both a **spend** and an **impressions** column (plus a CPM). There are also two
control variables: `price_index` and `competitor_spend`.

Key things to look for:
- **Variation across geos** — Do spending patterns differ by region? More variation = better identification.
- **Flighting patterns** — Some channels are always-on (digital, search), others are flighted (TV, display).
  Flighting creates on/off variation that helps separate media effects from trend.
- **Scale differences** — Revenue and spend levels may vary by region (population-driven).

In [ ]:
# --- Data Exploration ---
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# Revenue by geo over time
ax = axes[0, 0]
for geo in sorted(df['geo'].unique()):
    geo_df = df[df['geo'] == geo]
    ax.plot(pd.to_datetime(geo_df['date']), geo_df['revenue'], alpha=0.7, label=geo)
ax.set_title('Revenue by Region')
ax.legend(fontsize=7)
ax.tick_params(axis='x', rotation=45)

# Spend channels
spend_cols = ['tv_spend', 'digital_spend', 'social_spend', 'search_spend', 'display_spend']
for i, col in enumerate(spend_cols):
    ax = axes.flat[i + 1]
    for geo in sorted(df['geo'].unique()):
        geo_df = df[df['geo'] == geo]
        ax.plot(pd.to_datetime(geo_df['date']), geo_df[col], alpha=0.5, linewidth=0.8)
    ax.set_title(col.replace('_', ' ').title())
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Revenue and Media Spend by Region (5 geos × 104 weeks)', fontsize=13)
plt.tight_layout()
plt.show()

# Summary stats per geo
print("\n=== Revenue Summary by Geo ===")
print(df.groupby('geo')['revenue'].describe().round(1).to_string())

## Data Preparation for Meridian

Meridian's `InputData` expects 3D arrays shaped `(n_geos, n_times, n_channels)`. With multi-geo
data, the setup differs from Session 5 in several important ways:

- **Geos are a real dimension** — not a dummy `['national']` singleton.
- **Population matters** — Meridian uses population to scale KPI and media across regions of
  different sizes. We use approximate US Census region populations.
- **Spend and volume are separate** — Each channel has a natural spend ↔ impressions pairing.
  No need for spend-as-proxy hacks.
- **Controls** — `price_index` and `competitor_spend` are external factors, not marketing levers.

In [ ]:
# --- Data Preparation ---
date_col = 'date'
kpi_col = 'revenue'
geo_col = 'geo'

# Media channels: spend and volume (impressions) paired 1:1
media_spend_cols = ['tv_spend', 'digital_spend', 'social_spend', 'search_spend', 'display_spend']
media_volume_cols = ['tv_impressions', 'digital_impressions', 'social_impressions', 'search_impressions', 'display_impressions']
media_labels = ['TV', 'Digital', 'Social', 'Search', 'Display']

# Controls: non-actionable external factors
control_cols = ['price_index', 'competitor_spend']

# Sort by geo and date for consistent reshaping
df = df.sort_values([geo_col, date_col]).reset_index(drop=True)

geo_names = sorted(df[geo_col].unique())
time_values = sorted(df[date_col].unique())
n_geos = len(geo_names)
n_times = len(time_values)
n_media = len(media_spend_cols)
n_controls = len(control_cols)

print(f"Geos ({n_geos}): {geo_names}")
print(f"Time periods: {n_times} weeks")
print(f"Media channels ({n_media}): {media_labels}")
print(f"Controls ({n_controls}): {control_cols}")

# Reshape into (n_geos, n_times, n_channels)
# Pivot so each geo becomes a slice along axis 0
kpi_array = df.pivot(index=geo_col, columns=date_col, values=kpi_col).loc[geo_names, time_values].values

media_spend_array = np.stack([
    df.pivot(index=geo_col, columns=date_col, values=col).loc[geo_names, time_values].values
    for col in media_spend_cols
], axis=-1)  # (n_geos, n_times, n_media)

media_volume_array = np.stack([
    df.pivot(index=geo_col, columns=date_col, values=col).loc[geo_names, time_values].values
    for col in media_volume_cols
], axis=-1)

control_array = np.stack([
    df.pivot(index=geo_col, columns=date_col, values=col).loc[geo_names, time_values].values
    for col in control_cols
], axis=-1)

# Approximate US Census region populations (millions)
# Used by Meridian to scale KPI per capita across regions of different sizes
population_map = {
    'midwest': 68.99,
    'northeast': 57.61,
    'southeast': 84.29,
    'southwest': 44.38,
    'west': 78.59,
}
population_array = np.array([population_map[g] for g in geo_names])

# Verify shapes and data quality
print(f"\nFormatted shapes:")
print(f"  kpi:          {kpi_array.shape}")
print(f"  media_spend:  {media_spend_array.shape}")
print(f"  media_volume: {media_volume_array.shape}")
print(f"  controls:     {control_array.shape}")
print(f"  population:   {population_array.shape}")

print(f"\nData quality:")
print(f"  NaN: {np.isnan(kpi_array).sum() + np.isnan(media_spend_array).sum() + np.isnan(control_array).sum()}")
print(f"  Inf: {np.isinf(media_spend_array).sum() + np.isinf(control_array).sum()}")
print(f"  Negative KPI: {(kpi_array < 0).sum()}")

## Building InputData

With multi-geo data, the `InputData` object becomes the core differentiator from Session 5.
Meridian uses the geographic dimension to:

- **Estimate geo-level coefficients** — Each region gets its own media coefficient, drawn from
  a shared national prior. This is the "hierarchical" in hierarchical Bayesian.
- **Population-scale the KPI** — Regions with larger populations naturally have higher revenue.
  Meridian adjusts for this so media effects are comparable across geos.
- **Identify media effects** — If TV spend is high in the northeast but low in the midwest
  during the same week, and revenue follows that pattern, the model can attribute it to TV
  rather than to a shared seasonal trend.

In [ ]:
# --- Build InputData ---
if MERIDIAN_AVAILABLE:
    time_coords = sorted(df[date_col].unique())  # already YYYY-MM-DD strings
    geo_coords = geo_names

    input_data = InputData(
        kpi=xr.DataArray(
            kpi_array, name='kpi',
            dims=['geo', 'time'],
            coords={'time': time_coords, 'geo': geo_coords}
        ),
        kpi_type='revenue',
        population=xr.DataArray(
            population_array, name='population',
            dims=['geo'],
            coords={'geo': geo_coords}
        ),
        media_spend=xr.DataArray(
            media_spend_array, name='media_spend',
            dims=['geo', 'time', 'media_channel'],
            coords={'time': time_coords, 'geo': geo_coords, 'media_channel': media_labels}
        ),
        media=xr.DataArray(
            media_volume_array, name='media',
            dims=['geo', 'media_time', 'media_channel'],
            coords={'media_time': time_coords, 'geo': geo_coords, 'media_channel': media_labels}
        ),
        controls=xr.DataArray(
            control_array, name='controls',
            dims=['geo', 'time', 'control_variable'],
            coords={'time': time_coords, 'geo': geo_coords, 'control_variable': control_cols}
        ),
    )
    print("InputData created successfully!")
    print(f"  Geos:           {list(input_data.kpi.geo.values)}")
    print(f"  Media channels: {list(input_data.media_spend.media_channel.values)}")
    print(f"  Controls:       {list(input_data.controls.control_variable.values)}")
    print(f"  Time periods:   {input_data.kpi.sizes['time']}")
    print(f"  Population:     {dict(zip(geo_coords, population_array))}")
else:
    print("Skipping InputData creation (Meridian not installed).")
    print("Arrays are formatted and ready — use Colab if needed.")

## Model Specification

The `ModelSpec` is similar to Session 5 but with one crucial difference: **`max_lag` is now
measured in weeks, not months**. A `max_lag=8` here means 8 weeks of carry-over (≈2 months),
which is appropriate for weekly data.

For the hierarchical structure, Meridian's defaults work well:
- **Media coefficients**: Half-normal priors (positive ROI assumption)
- **Geo-level effects**: Each geo's beta is drawn from a shared national distribution
- **Adstock decay**: Beta prior on `alpha_m` — weekly decay is typically faster than monthly
- **Hill saturation**: LogNormal priors on `ec_m` and `slope_m`

With 520 observations (vs. 36 in Session 5), the data can inform these parameters much more
strongly — you'll see tighter posteriors and less dependence on priors.

In [ ]:
# --- Model Specification ---
if MERIDIAN_AVAILABLE:
    model_spec = meridian_spec.ModelSpec(
        max_lag=8,  # 8 weeks of carry-over effects
    )

    print("ModelSpec created.")
    print(f"  max_lag: {model_spec.max_lag} weeks")
    print(f"\nWith {n_geos} geos and {n_times} weeks, Meridian will estimate:")
    print(f"  - {n_geos} geo-level media coefficients per channel (shared prior)")
    print(f"  - {n_media} national adstock decay parameters")
    print(f"  - {n_media} national Hill saturation curves")
    print(f"  - {n_controls} control variable coefficients")
else:
    print("ModelSpec would use max_lag=8 (weeks of carry-over).")
    print("The geo-level hierarchy is enabled automatically when n_geos > 1.")

### Note: Model fitting with geo data

Multi-geo models are more complex — the sampler must explore a higher-dimensional parameter
space (geo-level coefficients for each channel). Expect fitting to take **10-30 minutes on CPU**.

On the flip side, the model has dramatically more data to learn from (520 vs 36 observations),
which typically improves convergence. The NUTS sampler should find the posterior geometry more
easily because the likelihood surface is better-defined.

**Tips:**
- On Google Colab with GPU, fitting drops to 2-5 minutes
- Start with `n_keep=500` for exploration; increase to 1000+ for final results
- Monitor R-hat — with 520 observations, convergence issues usually signal model misspecification,
  not insufficient sampling

In [ ]:
# --- Model Fitting ---
if MERIDIAN_AVAILABLE:
    # Initialize model
    mmm = meridian_model.Meridian(
        input_data=input_data,
        model_spec=model_spec,
    )

    print("Starting MCMC sampling (geo-level model)...")
    print(f"Parameters: {n_geos} geos × {n_media} channels × {n_times} weeks")
    print("This may take 10-30 minutes on CPU.\n")

    mmm.sample_posterior(
        n_chains=2,      # parallel MCMC chains
        n_adapt=500,     # adaptation steps
        n_burnin=500,    # warm-up iterations per chain
        n_keep=500,      # posterior samples to keep per chain
        seed=0,
    )

    print("\nModel fitting complete!")
else:
    print("Skipping model fitting (Meridian not installed).")
    print("Use Google Colab for GPU-accelerated fitting.")

## Convergence Diagnostics

With multi-geo models, convergence diagnostics become even more important. We now have
**geo-level parameters** (e.g., `beta_gm` has shape `(n_geos, n_media)`), so there are more
things that can go wrong. Check:

- **R-hat < 1.1** for all parameters — if any geo-channel combination has high R-hat, it may
  indicate that the data in that geo is insufficient to identify that channel's effect.
- **Effective sample size > 100** — low ESS in geo-level parameters can indicate that the
  hierarchical prior is too tight or too loose.
- **No divergent transitions** — divergences in a geo model often point to problematic
  geo-channel combinations (e.g., a channel with zero spend in one region).

In [ ]:
# --- Convergence Diagnostics ---
if MERIDIAN_AVAILABLE:
    import arviz as az

    inference_data = mmm.inference_data
    posterior = inference_data.posterior

    print("Inference data groups:", list(inference_data.groups()))
    print(f"\nPosterior variables: {list(posterior.data_vars)}")

    # Compute summary statistics
    summary = az.summary(inference_data, var_names=['beta_gm', 'alpha_m', 'ec_m', 'slope_m'])
    print(f"\nParameter summary (first 20 rows):")
    print(summary.head(20).to_string())

    # Check convergence
    bad_rhat = summary[summary['r_hat'] > 1.1]
    if len(bad_rhat) > 0:
        print(f"\nWARNING: {len(bad_rhat)} parameters with R-hat > 1.1:")
        print(bad_rhat[['mean', 'r_hat']].to_string())
    else:
        print(f"\nAll {len(summary)} parameters have R-hat < 1.1 — good convergence!")
else:
    print("Convergence diagnostics require a fitted model.")

## ROI Extraction

With geo-level data, Meridian can compute ROI at multiple levels:
- **National ROI** — aggregated across all geos (comparable to Session 5)
- **Geo-level ROI** — how each channel performs in each region

Geo-level ROI reveals insights invisible in national data. For example, TV might have
high ROI in the midwest (where it's the dominant media) but low ROI in the west (where
digital is more effective). These insights drive **geo-targeted budget allocation** —
shifting spend to channels and regions where it works hardest.

In [ ]:
# --- ROI Extraction ---
if MERIDIAN_AVAILABLE:
    # Sample prior (required by Analyzer)
    mmm.sample_prior(n_draws=500, seed=0)
    analyzer = Analyzer(mmm)

    try:
        # National-level summary metrics
        metrics = analyzer.summary_metrics()
        print("=== Summary Metrics ===")
        print(metrics.to_string())
    except Exception as e:
        print(f"Summary metrics error: {e}")

    try:
        # ROI posterior
        roi = analyzer.roi(use_posterior=True)
        roi_np = roi.numpy() if hasattr(roi, 'numpy') else np.array(roi)
        roi_mean = roi_np.mean(axis=(0, 1))  # mean across chains and draws

        print("\n=== ROI Posterior (mean across chains & draws) ===")
        for i, ch in enumerate(media_labels):
            print(f"  {ch}: {roi_mean[i]:.4f}")
    except Exception as e:
        print(f"ROI extraction error: {e}")
else:
    print("ROI extraction requires a fitted model.")

## Response Curves

Response curves show the **diminishing returns** relationship between spend and incremental
outcome for each channel. With geo-level data, these curves are estimated more precisely because:

- Each geo contributes a different point on the curve (regions that spend more are further
  along the saturation curve)
- The model sees natural spend variation across geos, helping pin down where the curve bends

Compare these curves to Session 5 — you should see **tighter confidence bands** reflecting
the larger sample size.

In [ ]:
# --- Response Curves ---
if MERIDIAN_AVAILABLE:
    try:
        from meridian.analysis import visualizer

        media_effects = visualizer.MediaEffects(mmm)
        chart = media_effects.plot_response_curves(confidence_level=0.9, include_ci=True)
        chart.display()
    except Exception as e:
        print(f"Response curve error: {e}")
        import traceback; traceback.print_exc()
else:
    print("Response curves require a fitted model.")

## Media Contributions Over Time

The contribution decomposition breaks down the KPI into the incremental effect of each
media channel over time. With geo-level data, contributions are estimated per-geo and can
be aggregated nationally, giving a more granular picture of what's driving revenue.

In [ ]:
# --- Contribution Over Time ---
if MERIDIAN_AVAILABLE:
    try:
        from meridian.analysis import visualizer

        media_summary = visualizer.MediaSummary(mmm)
        chart = media_summary.plot_channel_contribution_area_chart(time_granularity='weekly')
        chart.display()

        print("\n=== Contribution Waterfall ===")
        waterfall = media_summary.plot_contribution_waterfall_chart()
        waterfall.display()
    except Exception as e:
        print(f"Contribution error: {e}")
        import traceback; traceback.print_exc()
else:
    print("Contribution analysis requires a fitted model.")

## Posterior Distributions

With geo-level data, the posterior distributions are a key diagnostic. Compare these to
Session 5's posteriors — you should see:

- **Narrower posteriors** — 520 observations constrain parameters much more than 36
- **Separated channels** — With more data, the model can better distinguish between channels
  that looked similar with limited data
- **Geo-level `beta_gm`** — Media coefficients now have a geo dimension, so you can examine
  how each region responds to each channel

In [ ]:
# --- Posterior Distributions ---
if MERIDIAN_AVAILABLE:
    try:
        import arviz as az
        posterior = mmm.inference_data.posterior

        fig, axes = plt.subplots(2, 2, figsize=(12, 8))

        # 1. Media coefficients (national average across geos)
        ax = axes[0, 0]
        if 'beta_gm' in posterior:
            for i in range(n_media):
                # Average across geos for national view
                samples = posterior['beta_gm'].isel(media_channel=i).values.flatten()
                ax.hist(samples, bins=30, alpha=0.5, label=media_labels[i], density=True)
        ax.set_title('Media Coefficients (beta_gm)')
        ax.legend(fontsize=7)

        # 2. Hill half-saturation
        ax = axes[0, 1]
        if 'ec_m' in posterior:
            for i in range(n_media):
                samples = posterior['ec_m'].isel(media_channel=i).values.flatten()
                ax.hist(samples, bins=30, alpha=0.5, label=media_labels[i], density=True)
        ax.set_title('Hill Half-Saturation (ec_m)')
        ax.legend(fontsize=7)

        # 3. Hill slope
        ax = axes[1, 0]
        if 'slope_m' in posterior:
            for i in range(n_media):
                samples = posterior['slope_m'].isel(media_channel=i).values.flatten()
                ax.hist(samples, bins=30, alpha=0.5, label=media_labels[i], density=True)
        ax.set_title('Hill Slope (slope_m)')
        ax.legend(fontsize=7)

        # 4. Adstock decay
        ax = axes[1, 1]
        if 'alpha_m' in posterior:
            for i in range(n_media):
                samples = posterior['alpha_m'].isel(media_channel=i).values.flatten()
                ax.hist(samples, bins=30, alpha=0.5, label=media_labels[i], density=True)
        ax.set_title('Adstock Decay (alpha_m)')
        ax.legend(fontsize=7)

        plt.suptitle('Posterior Distributions (Geo-Level Model)', fontsize=13)
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Posterior visualization error: {e}")
        import traceback; traceback.print_exc()
else:
    print("Posterior distributions require a fitted model.")

## Geo-Level Analysis: Regional Media Coefficients

This is the analysis that **only geo-level data can provide**. The `beta_gm` parameter has
shape `(n_geos, n_media)`, giving each region its own media coefficient. This reveals:

- Which channels work best in which regions
- Whether national ROI averages hide important regional differences
- Opportunities for geo-targeted budget reallocation

For example, if TV's coefficient is 3× higher in the midwest than the northeast, the
brand should consider shifting TV budget toward midwest markets.

In [ ]:
# --- Geo-Level Media Coefficients ---
if MERIDIAN_AVAILABLE:
    try:
        posterior = mmm.inference_data.posterior

        if 'beta_gm' in posterior:
            # beta_gm shape: (chain, draw, geo, media_channel)
            beta = posterior['beta_gm'].mean(dim=['chain', 'draw']).values  # (n_geos, n_media)

            fig, ax = plt.subplots(figsize=(10, 6))
            x = np.arange(n_media)
            width = 0.15

            for g_idx, geo in enumerate(geo_names):
                ax.bar(x + g_idx * width, beta[g_idx], width, label=geo, alpha=0.8)

            ax.set_xticks(x + width * (n_geos - 1) / 2)
            ax.set_xticklabels(media_labels)
            ax.set_ylabel('Media Coefficient (posterior mean)')
            ax.set_title('Media Effectiveness by Region')
            ax.legend()
            plt.tight_layout()
            plt.show()

            # Print table
            print("\n=== Geo-Level Media Coefficients (posterior mean) ===")
            coef_df = pd.DataFrame(beta, index=geo_names, columns=media_labels)
            print(coef_df.round(4).to_string())
        else:
            print("beta_gm not found in posterior. Available:", list(posterior.data_vars))
    except Exception as e:
        print(f"Geo-level analysis error: {e}")
        import traceback; traceback.print_exc()
else:
    print("Geo-level analysis requires a fitted model.")

## Prior vs. Posterior Comparison

Comparing priors to posteriors tells us **how informative the data was**. With 520
geo-weekly observations (vs. 36 monthly), we expect the posteriors to be:

- **Much narrower** than the priors — the data is strongly informative
- **Shifted from the prior mean** — the data is overriding the default assumptions
- **Well-separated across channels** — enough data to distinguish channel effects

If a posterior still looks like its prior, that channel may lack sufficient variation
in the data (e.g., a channel with near-constant spend across all geos and weeks).

In [ ]:
# --- Prior vs Posterior Comparison ---
if MERIDIAN_AVAILABLE:
    try:
        prior = mmm.inference_data.prior
        posterior = mmm.inference_data.posterior

        prior_key = 'beta_m'
        post_key = 'beta_gm'

        n_plot = min(n_media, 6)
        fig, axes = plt.subplots(1, n_plot, figsize=(4 * n_plot, 4))
        if n_plot == 1:
            axes = [axes]

        for i in range(n_plot):
            ax = axes[i]
            channel = media_labels[i]

            if prior_key in prior:
                prior_samples = prior[prior_key].isel(media_channel=i).values.flatten()
                ax.hist(prior_samples, bins=30, alpha=0.4, color='gray', label='Prior', density=True)

            if post_key in posterior:
                post_samples = posterior[post_key].isel(media_channel=i).values.flatten()
                ax.hist(post_samples, bins=30, alpha=0.6, color='steelblue', label='Posterior', density=True)

            ax.set_title(channel)
            ax.set_xlabel('Coefficient')
            ax.legend(fontsize=7)

        plt.suptitle('Prior vs. Posterior: Media Coefficients (Geo Model)', fontsize=13)
        plt.tight_layout()
        plt.show()

        print("\nInterpretation:")
        print("  - Narrow posteriors far from prior → data is highly informative")
        print("  - Posterior ≈ prior → insufficient variation for that channel")
        print("  - Compare band widths to Session 5 — geo data should give tighter estimates")
    except Exception as e:
        print(f"Prior/posterior comparison error: {e}")
        import traceback; traceback.print_exc()
else:
    print("Prior vs. posterior comparison requires a fitted model.")

## HTML Summary Report

Meridian's `Summarizer` generates a self-contained HTML report with model fit statistics,
contributions, ROI, and budget allocation recommendations. With geo-level data, the report
includes regional breakdowns.

In [ ]:
# --- Generate HTML Summary ---
if MERIDIAN_AVAILABLE:
    from meridian.analysis.summarizer import Summarizer

    try:
        summary = Summarizer(mmm)

        start_date = df[date_col].min()
        end_date = df[date_col].max()
        output_dir = '.'
        filename = 'meridian_geo_summary'

        summary.output_model_results_summary(
            filename=filename,
            filepath=output_dir,
            start_date=start_date,
            end_date=end_date,
        )
        print(f"HTML summary saved to: {output_dir}/{filename}.html")
        print("Open in your browser to explore interactive results.")
    except Exception as e:
        print(f"Summarizer error: {e}")
        import traceback; traceback.print_exc()
else:
    print("HTML summary requires a fitted model.")

## Comparison: National Monthly vs. Geo Weekly

Here is a summary of the key differences between the two approaches:

| Aspect | Session 5 (National Monthly) | This Notebook (Geo Weekly) |
|--------|----------------------------|---------------------------|
| **Data points** | 36 | 520 |
| **Posterior width** | Wide (data-starved) | Narrow (data-rich) |
| **Adstock precision** | Monthly granularity hides fast decay | Weekly granularity captures true decay |
| **Geo insights** | None — single national number | Per-region media coefficients and ROI |
| **Causal strength** | Relies on time-series variation only | Geo × time variation aids identification |
| **Fitting time** | 5-15 min | 10-30 min |
| **Budget optimization** | National only | Can optimize across geos and channels |

### Key takeaways

1. **More data = better estimation.** The fundamental advantage of geo-weekly data is volume.
   With 14× more observations, the model has enough information to estimate complex
   relationships (adstock, saturation, geo effects) without over-relying on priors.

2. **Geographic variation is quasi-experimental.** When TV spend varies across regions in the
   same week, the model can use cross-sectional variation to identify the TV effect — similar
   to how a controlled experiment compares treatment and control groups.

3. **Geo-level insights drive better decisions.** Knowing that digital performs 2× better in
   the west than the midwest is actionable — it leads directly to geo-targeted budget allocation.
   National averages hide these opportunities.

4. **Weekly adstock is more realistic.** Most media effects decay over days to weeks, not months.
   Monthly data forces the model to aggregate these dynamics, losing the ability to distinguish
   fast-acting channels (search, social) from slow-acting ones (TV, outdoor).

### When is national monthly data acceptable?

- When geo-level data is genuinely unavailable (some markets don't report sub-national metrics)
- For quick directional estimates during early-stage planning
- When the brand operates in a single market with no geographic variation
- As a sanity check against the geo model (results should be roughly consistent)

## Exercises

1. **Compare ROI rankings** — Do the channel rankings from this geo model match Session 5's
   national model? Where do they disagree, and why?

2. **Examine geo-level coefficients** — Which region has the highest TV coefficient? The lowest?
   Does this make sense given the data (check spend levels and revenue by region)?

3. **Try `max_lag=4` vs `max_lag=12`** — With weekly data, adstock lag matters more. How do
   the decay curves change? Which channels are most sensitive to this setting?

4. **Budget reallocation** — If you could shift 20% of total media budget across channels and
   geos, where would the geo-level ROIs suggest you move it?

In [ ]:
# TODO: Exercise 1 - Compare ROI rankings to Session 5
# Load or reference Session 5 ROIs and compare channel rankings

# session_5_roi = {...}  # from Session 5
# geo_roi = {...}        # from this notebook
# Compare rankings

In [ ]:
# TODO: Exercise 2 - Analyze geo-level coefficients
# Which geo has highest/lowest TV coefficient?
# Cross-reference with spend and revenue levels

In [ ]:
# TODO: Exercise 3 - Try different max_lag values

# model_spec_lag4 = meridian_spec.ModelSpec(max_lag=4)
# mmm_lag4 = meridian_model.Meridian(input_data=input_data, model_spec=model_spec_lag4)
# mmm_lag4.sample_posterior(n_chains=2, n_adapt=500, n_burnin=500, n_keep=500, seed=1)

In [ ]:
# TODO: Exercise 4 - Budget reallocation analysis
# Use geo-level ROIs to propose a reallocation strategy
# Which channels/geos should get more budget? Which should get less?

## Deliverable

By the end of this session, you should have:

- A **fitted geo-level Meridian model** on 5 regions × 104 weeks of data
- **National and geo-level ROI** for each media channel
- **Response curves** with tighter confidence bands than the national monthly model
- **Geo-level media coefficients** showing regional effectiveness differences
- An understanding of **when and why** geo-weekly data produces better MMM results
- A basis for **geo-targeted budget optimization** in Session 6